# Fine-Tune NLLB-200-distilled-600M with LoRA & Flash Attention

This notebook fine-tunes [facebook/nllb-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M)
for any translation task using **LoRA** (parameter-efficient fine-tuning) and **SDPA / Flash Attention**.

## What you need
- A **CSV file** with two columns: `source` and `target` (your parallel translation data)
- A **GPU runtime** (Colab T4 or better)

## How to customize
1. Upload your CSV data and set `data_path` in the config
2. Set `src_lang` and `tgt_lang` to your [NLLB language codes](https://github.com/facebookresearch/flores/blob/main/flores200/README.md#languages-in-flores-200)
3. Adjust hyperparameters as needed
4. Run all cells

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece protobuf peft accelerate

## 2. Imports & GPU Setup

In [ ]:
import gc
import json
import os
import random
from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Tuple

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

import warnings
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

if IN_COLAB:
    drive.mount("/content/drive")

## 3. Configuration

**Edit the fields below to match your task.** At minimum, set:
- `data_path` — path to your CSV with `source` and `target` columns
- `src_lang` / `tgt_lang` — [NLLB language codes](https://github.com/facebookresearch/flores/blob/main/flores200/README.md#languages-in-flores-200)
- `output_dir` — where to save checkpoints and results

In [ ]:
@dataclass
class Config:
    # === Data ===
    data_path: str = "/content/drive/MyDrive/data/translation_data.csv"  # CSV with 'source' and 'target' columns
    output_dir: str = "/content/drive/MyDrive/nllb-finetune-output"
    test_size: float = 0.1          # fraction held out for val + test
    split_seed: int = 42

    # === Model ===
    model_name: str = "facebook/nllb-200-distilled-600M"
    src_lang: str = "eng_Latn"      # source language code
    tgt_lang: str = "sin_Sinh"      # target language code
    max_source_length: int = 512
    max_target_length: int = 256

    # === Generation ===
    generation_max_new_tokens: int = 256
    generation_num_beams: int = 2
    generation_batch_size: int = 64

    # === Training ===
    per_device_train_batch_size: int = 48
    gradient_accumulation_steps: int = 1
    per_device_eval_batch_size: int = 48
    learning_rate: float = 2e-4
    num_train_epochs: int = 8
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    lr_scheduler_type: str = "linear"
    logging_steps: int = 100
    eval_steps: int = 100
    save_steps: int = 100
    seed: int = 42

    # === LoRA ===
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    target_modules: Tuple[str, ...] = ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2")

    # === Performance ===
    use_flash_attention: bool = True    # uses SDPA (PyTorch scaled dot-product attention)
    use_torch_compile: bool = False     # experimental; may speed up training on Ampere+ GPUs

    # === Subset (for quick test runs) ===
    subset_ratio: float = 1.0  # set < 1.0 to train on a fraction of data


CONFIG = Config()

print(json.dumps(asdict(CONFIG), indent=2))

## 4. Helpers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_tokenizer(cfg: Config) -> Any:
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    tokenizer.src_lang = cfg.src_lang
    tokenizer.tgt_lang = cfg.tgt_lang
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def get_forced_bos_token_id(tokenizer: Any, cfg: Config) -> Optional[int]:
    """Resolve the token ID that forces generation in the target language."""
    token_id = tokenizer.convert_tokens_to_ids(cfg.tgt_lang)
    if token_id is None or int(token_id) < 0:
        lang_code_to_id = getattr(tokenizer, "lang_code_to_id", None)
        if isinstance(lang_code_to_id, dict):
            token_id = lang_code_to_id.get(cfg.tgt_lang)
    if token_id is None:
        return None
    token_id = int(token_id)
    return token_id if token_id >= 0 else None

## 5. Load & Split Data

Expects a CSV with `source` and `target` columns. The data is split into train / validation / test sets.

In [ ]:
def load_and_split_data(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(cfg.data_path)
    assert "source" in df.columns and "target" in df.columns, \
        "CSV must have 'source' and 'target' columns"

    df = df.dropna(subset=["source", "target"])
    df = df[df["source"].str.strip() != ""]
    df = df[df["target"].str.strip() != ""]
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df)} valid sentence pairs")

    train_df, temp_df = train_test_split(df, test_size=cfg.test_size * 2, random_state=cfg.split_seed)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=cfg.split_seed)

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"Split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split_data(CONFIG)

## 6. Tokenize

In [ ]:
def _normalize_token_ids(values: Any) -> List[int]:
    """Flatten token ID sequences from the tokenizer into a clean list of ints."""
    if values is None:
        return []
    if hasattr(values, "tolist"):
        values = values.tolist()
    while isinstance(values, (list, tuple)) and len(values) == 1 and isinstance(values[0], (list, tuple)):
        values = values[0]
    if not isinstance(values, (list, tuple)):
        return []
    return [int(x) for x in values if not isinstance(x, (list, tuple))]


def prepare_dataset(
    split_df: pd.DataFrame,
    tokenizer: Any,
    cfg: Config,
    split_name: str,
) -> Dataset:
    dataset = Dataset.from_pandas(split_df[["source", "target"]], preserve_index=False)

    def tokenize(example: Dict[str, Any]) -> Dict[str, Any]:
        src_tok = tokenizer(
            text=str(example["source"]),
            truncation=True,
            max_length=cfg.max_source_length,
            padding=False,
            add_special_tokens=True,
        )
        tgt_tok = tokenizer(
            text_target=str(example["target"]),
            truncation=True,
            max_length=cfg.max_target_length,
            padding=False,
            add_special_tokens=True,
        )

        input_ids = _normalize_token_ids(src_tok.get("input_ids"))
        attention_mask = _normalize_token_ids(src_tok.get("attention_mask"))
        labels = _normalize_token_ids(tgt_tok.get("input_ids"))

        if not input_ids:
            fallback = tokenizer.eos_token_id or 0
            input_ids = [int(fallback)]
        if len(attention_mask) != len(input_ids):
            attention_mask = [1] * len(input_ids)
        if not labels:
            fallback = tokenizer.eos_token_id or 0
            labels = [int(fallback)]

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

    prepared = dataset.map(tokenize, remove_columns=dataset.column_names)
    print(f"Tokenized {split_name}: {len(prepared)} examples")
    return prepared


tokenizer = load_tokenizer(CONFIG)
train_dataset = prepare_dataset(train_df, tokenizer, CONFIG, "train")
val_dataset = prepare_dataset(val_df, tokenizer, CONFIG, "val")

## 7. Load Model with LoRA & Flash Attention

In [ ]:
def load_model(cfg: Config, tokenizer: Any) -> Any:
    dtype = torch.float32
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    model_kwargs: Dict[str, Any] = {"torch_dtype": dtype}
    if cfg.use_flash_attention:
        model_kwargs["attn_implementation"] = "sdpa"

    try:
        model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name, **model_kwargs)
    except Exception:
        model_kwargs.pop("attn_implementation", None)
        model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name, **model_kwargs)
        print("SDPA not available, using default attention.")

    # Set forced BOS token for target language generation
    forced_bos_token_id = get_forced_bos_token_id(tokenizer, cfg)
    if forced_bos_token_id is None:
        raise RuntimeError(f"Could not resolve forced_bos_token_id for: {cfg.tgt_lang}")

    model.config.use_cache = False
    if hasattr(model.config, "forced_bos_token_id"):
        model.config.forced_bos_token_id = None
    if model.generation_config is not None:
        model.generation_config.forced_bos_token_id = forced_bos_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    # Apply LoRA
    from peft import LoraConfig, TaskType, get_peft_model

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    model = get_peft_model(model, lora_config)

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    model.config.use_cache = False
    if model.generation_config is not None:
        model.generation_config.forced_bos_token_id = forced_bos_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    if cfg.use_torch_compile and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
            print("torch.compile applied")
        except Exception as exc:
            print(f"torch.compile skipped: {exc}")

    model.print_trainable_parameters()
    return model


model = load_model(CONFIG, tokenizer)

## 8. Training

In [ ]:
class JsonlLogger(TrainerCallback):
    """Logs training metrics to a JSONL file."""
    def __init__(self, path: str):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        open(self.path, "w").close()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        record = {"step": int(state.global_step)}
        record.update({k: float(v) if isinstance(v, (int, float)) else v for k, v in logs.items()})
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")


def build_compute_metrics(tokenizer: Any):
    bleu_metric = evaluate.load("sacrebleu")
    ter_metric = evaluate.load("ter")
    chrf_metric = evaluate.load("chrf")

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]

        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        vocab_size = len(tokenizer)
        preds = np.clip(preds, 0, vocab_size - 1).astype(np.int32)
        labels = np.clip(labels, 0, vocab_size - 1).astype(np.int32)

        decoded_preds = [x.strip() for x in tokenizer.batch_decode(preds, skip_special_tokens=True)]
        decoded_labels = [x.strip() for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
        refs = [[l] for l in decoded_labels]

        valid = [(p, r) for p, r in zip(decoded_preds, refs) if p and r[0]]
        if not valid:
            return {"bleu": 0.0, "ter": 100.0, "chrf": 0.0}

        v_preds, v_refs = [x[0] for x in valid], [x[1] for x in valid]
        bleu = float(bleu_metric.compute(predictions=v_preds, references=v_refs)["score"])
        ter = float(ter_metric.compute(predictions=v_preds, references=v_refs)["score"])
        chrf = float(chrf_metric.compute(predictions=v_preds, references=v_refs, word_order=2)["score"])
        return {"bleu": bleu, "ter": ter, "chrf": chrf}

    return compute_metrics

In [ ]:
set_seed(CONFIG.seed)
os.makedirs(CONFIG.output_dir, exist_ok=True)

# Optional: use a subset for quick test runs
if CONFIG.subset_ratio < 1.0:
    n_train = max(1, int(len(train_dataset) * CONFIG.subset_ratio))
    n_val = max(1, int(len(val_dataset) * CONFIG.subset_ratio))
    rng = np.random.default_rng(CONFIG.seed)
    train_dataset = train_dataset.select(sorted(rng.permutation(len(train_dataset))[:n_train].tolist()))
    val_dataset = val_dataset.select(sorted(rng.permutation(len(val_dataset))[:n_val].tolist()))
    print(f"Using subset: train={len(train_dataset)}, val={len(val_dataset)}")

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt",
    label_pad_token_id=-100,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(CONFIG.output_dir, "checkpoints"),
    per_device_train_batch_size=CONFIG.per_device_train_batch_size,
    per_device_eval_batch_size=CONFIG.per_device_eval_batch_size,
    gradient_accumulation_steps=CONFIG.gradient_accumulation_steps,
    warmup_ratio=CONFIG.warmup_ratio,
    num_train_epochs=CONFIG.num_train_epochs,
    learning_rate=CONFIG.learning_rate,
    optim="adamw_torch_fused",
    weight_decay=CONFIG.weight_decay,
    max_grad_norm=CONFIG.max_grad_norm,
    lr_scheduler_type=CONFIG.lr_scheduler_type,
    logging_steps=CONFIG.logging_steps,
    eval_strategy="steps",
    eval_steps=CONFIG.eval_steps,
    save_strategy="steps",
    save_steps=CONFIG.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="ter",
    greater_is_better=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=CONFIG.seed,
    data_seed=CONFIG.seed,
    remove_unused_columns=True,
    predict_with_generate=True,
    generation_max_length=CONFIG.max_target_length,
    generation_num_beams=CONFIG.generation_num_beams,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    compute_metrics=build_compute_metrics(tokenizer),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        JsonlLogger(os.path.join(CONFIG.output_dir, "train_log.jsonl")),
    ],
)

trainer.train()

## 9. Save the Fine-Tuned Model

In [ ]:
final_model_dir = os.path.join(CONFIG.output_dir, "final_model")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"Model saved to: {final_model_dir}")

# Clean up training objects
del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 10. Evaluate on Test Set

In [ ]:
def load_finetuned_model(cfg: Config, adapter_path: str) -> Tuple[Any, Any]:
    """Load the base model with the fine-tuned LoRA adapter for inference."""
    tokenizer = load_tokenizer(cfg)
    dtype = torch.float32
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    base_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name, torch_dtype=dtype)

    from peft import PeftModel
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    model.config.use_cache = True

    forced_bos_token_id = get_forced_bos_token_id(tokenizer, cfg)
    if forced_bos_token_id is not None and model.generation_config is not None:
        model.generation_config.forced_bos_token_id = forced_bos_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    return model, tokenizer


def translate_batch(
    sources: List[str],
    model: Any,
    tokenizer: Any,
    cfg: Config,
) -> List[str]:
    """Translate a list of source texts in batches."""
    predictions: List[str] = []
    forced_bos_token_id = get_forced_bos_token_id(tokenizer, cfg)

    for i in tqdm(range(0, len(sources), cfg.generation_batch_size), desc="Translating"):
        batch = sources[i : i + cfg.generation_batch_size]
        inputs = tokenizer(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=cfg.max_source_length,
            add_special_tokens=True,
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.generation_max_new_tokens,
                do_sample=False,
                num_beams=cfg.generation_num_beams,
                forced_bos_token_id=forced_bos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([t.strip() for t in decoded])

    return predictions

In [ ]:
eval_model, eval_tokenizer = load_finetuned_model(CONFIG, final_model_dir)

sources = test_df["source"].astype(str).tolist()
references = test_df["target"].astype(str).tolist()
predictions = translate_batch(sources, eval_model, eval_tokenizer, CONFIG)

# Compute metrics
bleu_metric = evaluate.load("sacrebleu")
ter_metric = evaluate.load("ter")
chrf_metric = evaluate.load("chrf")

refs = [[r] for r in references]
bleu = bleu_metric.compute(predictions=predictions, references=refs)["score"]
ter = ter_metric.compute(predictions=predictions, references=refs)["score"]
chrf = chrf_metric.compute(predictions=predictions, references=refs, word_order=2)["score"]

print(f"\n{'='*50}")
print(f"Test Results ({len(test_df)} samples)")
print(f"{'='*50}")
print(f"  BLEU:  {bleu:.2f}")
print(f"  TER:   {ter:.2f}")
print(f"  chrF2: {chrf:.2f}")
print(f"{'='*50}")

# Save predictions
results_df = test_df.copy()
results_df["prediction"] = predictions
results_df.to_csv(os.path.join(CONFIG.output_dir, "test_predictions.csv"), index=False)

metrics = {"bleu": bleu, "ter": ter, "chrf": chrf, "test_size": len(test_df)}
with open(os.path.join(CONFIG.output_dir, "test_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nResults saved to: {CONFIG.output_dir}")

# Clean up
del eval_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 11. Inference Example

Use the fine-tuned model to translate new text:

In [ ]:
model, tokenizer = load_finetuned_model(CONFIG, final_model_dir)

sample_texts = [
    "The committee approved the budget.",
    "Machine learning models require large amounts of data.",
]

translations = translate_batch(sample_texts, model, tokenizer, CONFIG)

for src, tgt in zip(sample_texts, translations):
    print(f"  {CONFIG.src_lang}: {src}")
    print(f"  {CONFIG.tgt_lang}: {tgt}")
    print()